In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Dataset 03 — NHANES (National Health and Nutrition Examination Survey)

**Descripción:** Encuesta nacional continua de salud y nutrición realizada por los Centros para el Control y Prevención de Enfermedades (CDC) de EE.UU. Combina entrevistas domiciliarias y exámenes físicos para evaluar el estado de salud y nutrición de adultos y niños.

**Tarea:** Clasificación o regresión — predicción de diabetes, obesidad, enfermedades cardiovasculares, etc.

**Características:**
- Decenas de miles de participantes por ciclo (ciclos bienales desde 1999)
- Variables: datos demográficos, resultados de laboratorio, examen físico, dieta, cuestionarios
- Variable objetivo común: `DIQ010` (diabetes), `BMXBMI` (IMC), `LBXGH` (hemoglobina A1c)
- Datos en formato SAS XPT o CSV descargables por módulos

**URL oficial:** https://www.cdc.gov/nchs/nhanes/index.html

**Módulos disponibles:**
- Demographics (DEMO)
- Examination (BMX — Body Measures)
- Laboratory (GHB — Glycohemoglobin, GLU — Glucose)
- Questionnaire (DIQ — Diabetes, BPQ — Blood Pressure)

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline
print('Librerías cargadas correctamente')

## 1. Carga del Dataset

Se cargan los módulos principales del ciclo NHANES 2017-2018 en formato CSV.

In [ ]:
# Cargar módulo demográfico
ruta_demo = '/content/drive/MyDrive/machine learning/datasets/nhanes/DEMO_J.csv'
ruta_bmx  = '/content/drive/MyDrive/machine learning/datasets/nhanes/BMX_J.csv'
ruta_diq  = '/content/drive/MyDrive/machine learning/datasets/nhanes/DIQ_J.csv'
ruta_ghb  = '/content/drive/MyDrive/machine learning/datasets/nhanes/GHB_J.csv'

demo = pd.read_csv(ruta_demo)
bmx  = pd.read_csv(ruta_bmx)
diq  = pd.read_csv(ruta_diq)
ghb  = pd.read_csv(ruta_ghb)

print(f'Demographics:   {demo.shape}')
print(f'Body Measures:  {bmx.shape}')
print(f'Diabetes Q:     {diq.shape}')
print(f'Glycohemoglobin:{ghb.shape}')

## 2. Unión de Módulos

In [ ]:
# Unir por SEQN (identificador único del participante)
df = demo.merge(bmx[['SEQN','BMXBMI','BMXWT','BMXHT']], on='SEQN', how='left')
df = df.merge(diq[['SEQN','DIQ010']], on='SEQN', how='left')
df = df.merge(ghb[['SEQN','LBXGH']], on='SEQN', how='left')

print(f'Dataset unido: {df.shape}')
print(f'Columnas: {list(df.columns)}')
df.head()

## 3. Valores Nulos

In [ ]:
nulos = df.isnull().sum()
print(nulos[nulos > 0].sort_values(ascending=False))
print(f'\nTotal nulos: {df.isnull().sum().sum()}')

## 4. Preprocesamiento con Pandas

In [ ]:
df_clean = df.copy()

# Seleccionar columnas relevantes
cols_usar = ['RIDAGEYR','RIAGENDR','RIDRETH3','DMDEDUC2','BMXBMI','BMXWT','BMXHT','LBXGH','DIQ010']
df_clean = df_clean[cols_usar].dropna(subset=['DIQ010'])

# DIQ010: 1=Sí diabetes, 2=No, 7=No sabe, 9=No respondió
df_clean = df_clean[df_clean['DIQ010'].isin([1,2])]
df_clean['diabetes'] = (df_clean['DIQ010'] == 1).astype(int)

# Imputar nulos restantes con mediana
for col in df_clean.columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print(f'Dataset limpio: {df_clean.shape}')
print(f'Con diabetes:    {df_clean["diabetes"].sum()} ({df_clean["diabetes"].mean()*100:.1f}%)')
print(f'Sin diabetes:    {(df_clean["diabetes"]==0).sum()} ({(df_clean["diabetes"]==0).mean()*100:.1f}%)')

## 5. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribución de IMC
axes[0].hist(df_clean['BMXBMI'].dropna(), bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Distribución de IMC')
axes[0].set_xlabel('IMC (kg/m²)')
axes[0].set_ylabel('Frecuencia')

# Distribución de Diabetes
axes[1].bar(['No Diabetes (0)', 'Diabetes (1)'],
            [int((df_clean['diabetes']==0).sum()), int(df_clean['diabetes'].sum())],
            color=['steelblue','tomato'], edgecolor='black')
axes[1].set_title('Distribución de Diabetes')
axes[1].set_ylabel('Número de participantes')

# IMC por Diabetes
for val, color, label in [(0,'steelblue','No Diabetes'), (1,'tomato','Diabetes')]:
    subset = df_clean[df_clean['diabetes']==val]['BMXBMI'].dropna()
    axes[2].hist(subset, bins=20, alpha=0.6, color=color, label=label, edgecolor='black')
axes[2].set_title('IMC por Diabetes')
axes[2].set_xlabel('IMC')
axes[2].legend()

plt.tight_layout()
plt.show()

## 6. División 80/20

In [ ]:
X_cols = ['RIDAGEYR','RIAGENDR','RIDRETH3','DMDEDUC2','BMXBMI','BMXWT','BMXHT','LBXGH']
X = df_clean[X_cols].to_numpy()
y = df_clean['diabetes'].to_numpy()

np.random.seed(42)
idx = np.random.permutation(len(X))
corte = int(len(X)*0.8)
X_train, X_test = X[idx[:corte]], X[idx[corte:]]
y_train, y_test = y[idx[:corte]], y[idx[corte:]]
print(f'Entrenamiento: {X_train.shape[0]} | Prueba: {X_test.shape[0]}')

## 7. Conclusiones

**NHANES** es una encuesta nacional de salud y nutrición que proporciona datos representativos de la población estadounidense. El dataset es ideal para tareas de **clasificación** (predicción de diabetes, hipertensión) y **regresión** (predicción de IMC, glucosa). Se identificó la prevalencia de diabetes en la muestra. Los módulos se unen por el identificador `SEQN`. El preprocesamiento incluyó filtrado de respuestas válidas y eliminación de valores especiales (7=No sabe, 9=No respondió).